# LeetCode 3753 — Total Waviness of Numbers in Range II bằng Python

- Link: https://leetcode.com/problems/total-waviness-of-numbers-in-range-ii/
- Độ khó: **Hard**
- Chủ đề: **Digit DP** (quy hoạch động theo chữ số) trả về cặp (số lượng, tổng), kỹ thuật `f(num2) − f(num1 − 1)`, đếm đóng góp

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def totalWaviness(self, num1: int, num2: int) -> int:
```

## 1. Mục tiêu học

1. Đưa bài đoạn `[num1, num2]` về hàm tiền tố `calc(x)` = tổng độ gợn sóng trên `[1, x]`.
2. Thiết kế trạng thái digit DP: `(pos, prev2, prev1, started, tight)` — chỉ hai chữ số cuối quyết định đỉnh/đáy.
3. DP trả về **cặp** `(cnt, sum)`: số cách hoàn thành và tổng đóng góp — cộng `add · cnt` khi đặt một chữ số tạo đỉnh/đáy.
4. Xử lý **số 0 đứng đầu** (`started`) để số có ít chữ số không bị tính sai.
5. Một góc nhìn khác: **đếm đóng góp** của từng bộ ba chữ số tại từng vị trí.

## 2. Đề bài & ví dụ

Độ gợn sóng của một số = số **đỉnh** (chữ số lớn hơn hẳn cả hai chữ số kề) + số **đáy** (nhỏ hơn hẳn cả hai). Chữ số đầu và cuối không tính; số có ít hơn 3 chữ số có độ gợn sóng 0. Tính tổng độ gợn sóng của mọi số trong `[num1, num2]`.

**Ví dụ 1:** `num1 = 120, num2 = 130` → `3` (`120`, `121`, `130`).

**Ví dụ 2:** `num1 = 198, num2 = 202` → `3` (`198`, `201`, `202`).

**Ví dụ 3:** `num1 = num2 = 4848` → `2`.

**Ràng buộc:** `1 ≤ num1 ≤ num2 ≤ 10^15`.

In [1]:
from functools import lru_cache
import random, time

examples = [(120, 130, 3), (198, 202, 3), (4848, 4848, 2)]
for a, b, exp in examples:
    print(a, b, "-> kỳ vọng", exp)

120 130 -> kỳ vọng 3
198 202 -> kỳ vọng 3
4848 4848 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Độ gợn sóng của một số

In [2]:
def waviness(x):
    d = list(map(int, str(x)))
    return sum(1 for i in range(1, len(d) - 1)
               if (d[i] > d[i - 1] and d[i] > d[i + 1]) or (d[i] < d[i - 1] and d[i] < d[i + 1]))

print([(x, waviness(x)) for x in [120, 121, 122, 4848, 13131, 99, 909]])

[(120, 1), (121, 1), (122, 0), (4848, 2), (13131, 3), (99, 0), (909, 1)]


### 3.2. Kỹ thuật tiền tố cho bài đoạn

`Σ_{x=num1}^{num2} w(x) = calc(num2) − calc(num1 − 1)` với `calc(x) = Σ_{y=1}^{x} w(y)` (và `calc(0) = 0`).

### 3.3. Khung digit DP

Đi từ chữ số cao nhất. `tight` = các chữ số đã đặt trùng tiền tố của `x` (chữ số tiếp theo bị chặn bởi chữ số tương ứng của `x`). `started` = đã đặt chữ số khác 0 đầu tiên chưa (các số 0 đứng đầu chỉ là "chưa bắt đầu", không phải chữ số thật). Khi `tight = False`, kết quả chỉ phụ thuộc phần còn lại ⇒ nhớ (memo) được.

In [3]:
# ví dụ khung đơn giản: đếm số trong [1, x] có tổng chữ số chẵn
def count_even_digit_sum(x):
    s = str(x)
    @lru_cache(None)
    def go(pos, parity, started, tight):
        if pos == len(s):
            return int(started and parity == 0)
        up = int(s[pos]) if tight else 9
        return sum(go(pos + 1, (parity + d) % 2, started or d > 0, tight and d == up) for d in range(up + 1))
    return go(0, 0, False, True)

print(count_even_digit_sum(100), sum(1 for y in range(1, 101) if sum(map(int, str(y))) % 2 == 0))

49 49


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Khi đặt chữ số `d` tại vị trí `pos`, chữ số **trước đó** (`prev1`) trở thành "chữ số giữa" có đủ hai hàng xóm `prev2` và `d`. Vậy:

- `add = 1` nếu `prev2` tồn tại và (`prev2 < prev1 > d` hoặc `prev2 > prev1 < d`).
- Mỗi cách hoàn thành phần còn lại đều được hưởng `add` này ⇒ tổng tăng `add · cnt(phần còn lại)`.

Do đó DP cần trả về **cả** số cách `cnt` lẫn tổng `wav`: `wav(cha) = Σ_d [wav(con) + add_d · cnt(con)]`.

## 5. Cách 1 — Brute Force

In [4]:
def brute(num1, num2):
    return sum(waviness(x) for x in range(num1, num2 + 1))

for a, b, exp in examples:
    print(brute(a, b), "kỳ vọng", exp)

3 kỳ vọng 3
3 kỳ vọng 3
2 kỳ vọng 2


**Nút thắt:** `O((num2 − num1) · 16)` — với `10^15` là bất khả thi.

## 6. Ý tưởng thuật toán / bước đột phá

`calc(x)` bằng digit DP với trạng thái `(pos, prev2, prev1, started, tight)`, dùng `10` làm giá trị "chưa có chữ số":

- Nếu chưa `started` và `d = 0`: vẫn chưa bắt đầu, `prev2 = prev1 = 10`.
- Nếu chưa `started` và `d > 0`: bắt đầu, `prev2 = 10, prev1 = d`.
- Nếu đã `started`: `prev2, prev1 ← prev1, d`; tính `add` như mục 4 (chỉ khi `prev2 ≠ 10`).

Số trạng thái: `16 · 11 · 11 · 2 · 2` ≈ 7.700, mỗi trạng thái thử 10 chữ số ⇒ rất nhanh.

## 7. Các cách tiếp theo

- **Cách 2:** digit DP đệ quy có memo trả về `(cnt, wav)` (lời giải nộp).
- **Cách 3:** digit DP **lặp** (tabulation) — duy trì bảng `(prev2, prev1) → (cnt, wav)` cho nhánh "tự do", tách riêng nhánh "tight".
- **Cách 4:** **đếm đóng góp**: với mỗi độ dài `L`, vị trí giữa `i`, bộ ba `(a, b, c)` là đỉnh/đáy, đếm số `≤ x` có đúng bộ ba đó tại `i−1, i, i+1`.

### 7.1. Cách 2 — Digit DP đệ quy (lời giải chính)

In [5]:
def calc_memo(x):
    if x <= 0:
        return 0
    s = str(x); n = len(s)

    @lru_cache(None)
    def dfs(pos, p2, p1, started, tight):
        if pos == n:
            return (1 if started else 0), 0
        up = int(s[pos]) if tight else 9
        cnt = wav = 0
        for d in range(up + 1):
            nt = tight and d == up
            if not started:
                if d == 0:
                    c, w = dfs(pos + 1, 10, 10, False, nt); add = 0
                else:
                    c, w = dfs(pos + 1, 10, d, True, nt); add = 0
            else:
                add = 1 if p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)) else 0
                c, w = dfs(pos + 1, p1, d, True, nt)
            cnt += c; wav += w + add * c
        return cnt, wav

    return dfs(0, 10, 10, False, True)[1]

def memo_range(a, b):
    return calc_memo(b) - calc_memo(a - 1)

for a, b, exp in examples:
    print(memo_range(a, b), "kỳ vọng", exp)

3 kỳ vọng 3
3 kỳ vọng 3
2 kỳ vọng 2


### 7.2. Cách 3 — Digit DP lặp (tabulation)

Duy trì `free[(p2, p1)] = (cnt, wav)` cho các tiền tố **đã nhỏ hơn** tiền tố của `x` (đã bắt đầu), và một tiền tố "tight" duy nhất. Ở mỗi vị trí:

- mọi trạng thái `free` mở rộng bằng mọi chữ số `0..9`;
- tiền tố tight mở rộng bằng `d < s[pos]` (sang `free`) và `d = s[pos]` (vẫn tight);
- số **bắt đầu tại vị trí này** (ngắn hơn `x`) với chữ số đầu `1..9` được thêm vào `free` nếu `pos > 0`.

In [6]:
def calc_tab(x):
    if x <= 0:
        return 0
    s = list(map(int, str(x))); n = len(s)

    def is_wave(p2, p1, d):
        return p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d))

    free = {}                                   # (p2, p1) -> [cnt, wav]
    tight_state = (10, s[0]); tight_wav = 0
    for d in range(1, s[0]):                    # chữ số đầu nhỏ hơn s[0]
        free[(10, d)] = [1, 0]
    for pos in range(1, n):
        nf = {}
        def put(key, c, w):
            if key in nf: nf[key][0] += c; nf[key][1] += w
            else: nf[key] = [c, w]
        for (p2, p1), (c, w) in free.items():
            for d in range(10):
                put((p1, d), c, w + c * is_wave(p2, p1, d))
        p2, p1 = tight_state
        for d in range(s[pos]):
            put((p1, d), 1, tight_wav + is_wave(p2, p1, d))
        tight_wav += is_wave(p2, p1, s[pos]); tight_state = (p1, s[pos])
        for d in range(1, 10):                  # số ngắn hơn bắt đầu tại pos
            put((10, d), 1, 0)
        free = nf
    return sum(w for c, w in free.values()) + tight_wav

def tab_range(a, b):
    return calc_tab(b) - calc_tab(a - 1)

for a, b, exp in examples:
    print(tab_range(a, b), "kỳ vọng", exp)

3 kỳ vọng 3
3 kỳ vọng 3
2 kỳ vọng 2


### 7.3. Cách 4 — Đếm đóng góp từng bộ ba

`calc(x) = Σ_{L} Σ_{i=1}^{L−2} Σ_{(a,b,c) sóng} #{y ≤ x, y có L chữ số, y[i−1..i+1] = (a,b,c)}`.

Đếm số có `L` chữ số với 3 chữ số cố định và `≤ x`:

- nếu `L < len(x)`: không bị chặn — chữ số đầu có 9 lựa chọn (hoặc cố định, phải khác 0), các chữ số tự do khác 10 lựa chọn;
- nếu `L = len(x)`: duyệt tiền tố tight như digit DP thường nhưng chữ số ở vị trí cố định chỉ có một lựa chọn.

In [7]:
WAVES = [(a, b, c) for a in range(10) for b in range(10) for c in range(10)
         if (b > a and b > c) or (b < a and b < c)]

def count_fixed(xs, L, i, triple):
    # số có đúng L chữ số, chữ số i-1, i, i+1 bằng triple, và <= x (xs = list chữ số của x)
    fixed = {i - 1: triple[0], i: triple[1], i + 1: triple[2]}
    if fixed.get(0, 1) == 0:
        return 0
    def choices(p):                              # số lựa chọn tự do cho vị trí p
        if p in fixed: return 1
        return 9 if p == 0 else 10
    if L < len(xs):
        r = 1
        for p in range(L): r *= choices(p)
        return r
    total = 0
    for p in range(L):                           # tiền tố trùng x tới p-1, chữ số p nhỏ hơn xs[p]
        lo = 1 if p == 0 else 0
        if p in fixed:
            less = 1 if lo <= fixed[p] < xs[p] else 0
        else:
            less = max(0, xs[p] - lo)
        if less:
            r = less
            for q in range(p + 1, L): r *= choices(q)
            total += r
        if p in fixed and fixed[p] != xs[p]:
            return total
    return total + 1                              # chính x

def calc_contrib(x):
    if x <= 0: return 0
    xs = list(map(int, str(x))); res = 0
    for L in range(3, len(xs) + 1):
        for i in range(1, L - 1):
            for t in WAVES:
                res += count_fixed(xs, L, i, t)
    return res

print([calc_contrib(b) - calc_contrib(a - 1) for a, b, _ in examples])

[3, 3, 2]


**Độ phức tạp Cách 4:** `O(len² · |WAVES| · len)` ≈ `16 · 16 · 570 · 16` ≈ 2,3 triệu phép — chậm hơn DP nhưng cho trực giác "mỗi bộ ba đóng góp độc lập".

## 8. TRACE từng bước

In [8]:
def trace_add(x):
    d = list(map(int, str(x)))
    print("số", x)
    p2, p1 = 10, 10; started = False; total = 0
    for pos, dig in enumerate(d):
        if not started:
            if dig: started = True; p1 = dig
            print(f"  pos {pos}: đặt {dig} -> bắt đầu, (p2, p1) = ({p2}, {p1})")
            continue
        add = int(p2 != 10 and ((p1 > p2 and p1 > dig) or (p1 < p2 and p1 < dig)))
        total += add
        print(f"  pos {pos}: đặt {dig}, giữa = {p1} giữa {p2} và {dig} -> add = {add}, (p2, p1) -> ({p1}, {dig})")
        p2, p1 = p1, dig
    print("  độ gợn sóng =", total)

trace_add(4848)
trace_add(13021)

số 4848
  pos 0: đặt 4 -> bắt đầu, (p2, p1) = (10, 4)
  pos 1: đặt 8, giữa = 4 giữa 10 và 8 -> add = 0, (p2, p1) -> (4, 8)
  pos 2: đặt 4, giữa = 8 giữa 4 và 4 -> add = 1, (p2, p1) -> (8, 4)
  pos 3: đặt 8, giữa = 4 giữa 8 và 8 -> add = 1, (p2, p1) -> (4, 8)
  độ gợn sóng = 2
số 13021
  pos 0: đặt 1 -> bắt đầu, (p2, p1) = (10, 1)
  pos 1: đặt 3, giữa = 1 giữa 10 và 3 -> add = 0, (p2, p1) -> (1, 3)
  pos 2: đặt 0, giữa = 3 giữa 1 và 0 -> add = 1, (p2, p1) -> (3, 0)
  pos 3: đặt 2, giữa = 0 giữa 3 và 2 -> add = 1, (p2, p1) -> (0, 2)
  pos 4: đặt 1, giữa = 2 giữa 0 và 1 -> add = 1, (p2, p1) -> (2, 1)
  độ gợn sóng = 3


In [9]:
# Trace pha tabulation cho x = 130: số trạng thái free và tổng sau mỗi vị trí
def trace_tab(x):
    s = list(map(int, str(x)))
    free = {(10, d): [1, 0] for d in range(1, s[0])}; tight_state = (10, s[0]); tight_wav = 0
    print(f"pos 0: free có {len(free)} trạng thái, tight = {tight_state}")
    for pos in range(1, len(s)):
        nf = {}
        def put(k, c, w):
            nf.setdefault(k, [0, 0]); nf[k][0] += c; nf[k][1] += w
        wv = lambda a, b, c: int(a != 10 and ((b > a and b > c) or (b < a and b < c)))
        for (p2, p1), (c, w) in free.items():
            for d in range(10): put((p1, d), c, w + c * wv(p2, p1, d))
        p2, p1 = tight_state
        for d in range(s[pos]): put((p1, d), 1, tight_wav + wv(p2, p1, d))
        tight_wav += wv(p2, p1, s[pos]); tight_state = (p1, s[pos])
        for d in range(1, 10): put((10, d), 1, 0)
        free = nf
        print(f"pos {pos}: free {len(free)} trạng thái, tổng cnt = {sum(c for c, _ in free.values())}, "
              f"tổng wav = {sum(w for _, w in free.values())}, tight = {tight_state} (wav {tight_wav})")
    print("calc =", sum(w for _, w in free.values()) + tight_wav, "| brute:", brute(1, x))

trace_tab(130)

pos 0: free có 0 trạng thái, tight = (10, 1)
pos 1: free 12 trạng thái, tổng cnt = 12, tổng wav = 0, tight = (1, 3) (wav 0)
pos 2: free 109 trạng thái, tổng cnt = 129, tổng wav = 11, tight = (3, 0) (wav 1)
calc = 12 | brute: 12


## 9. Code nộp LeetCode

In [10]:
from functools import lru_cache

class Solution:
    def totalWaviness(self, num1: int, num2: int) -> int:
        def calc(x: int) -> int:
            if x <= 0:
                return 0
            s = str(x)
            n = len(s)

            @lru_cache(None)
            def dfs(pos, p2, p1, started, tight):
                if pos == n:
                    return (1 if started else 0), 0
                up = int(s[pos]) if tight else 9
                cnt = wav = 0
                for d in range(up + 1):
                    nt = tight and d == up
                    if not started:
                        if d == 0:
                            c, w = dfs(pos + 1, 10, 10, False, nt)
                        else:
                            c, w = dfs(pos + 1, 10, d, True, nt)
                        cnt += c; wav += w
                    else:
                        add = 1 if p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)) else 0
                        c, w = dfs(pos + 1, p1, d, True, nt)
                        cnt += c; wav += w + add * c
                return cnt, wav

            return dfs(0, 10, 10, False, True)[1]

        return calc(num2) - calc(num1 - 1)

sol = Solution()
for a, b, exp in examples:
    got = sol.totalWaviness(a, b)
    print(a, b, "->", got, "✔" if got == exp else "✘")

120 130 -> 3 ✔
198 202 -> 3 ✔
4848 4848 -> 2 ✔


## 10. Vì sao thuật toán đúng

1. **Phân rã đóng góp.** `w(y) = Σ_i [y_i là đỉnh/đáy]`, và điều kiện tại `i` chỉ phụ thuộc `(y_{i−1}, y_i, y_{i+1})`. Khi đặt `y_{i+1}`, ta biết đủ ba chữ số ⇒ cộng đúng một lần cho mỗi vị trí giữa.
2. **Trạng thái đủ.** Tương lai chỉ cần hai chữ số cuối (để tính `add` cho bước kế tiếp), `started` (để biết chữ số có thật không) và `tight` (để biết giới hạn) — mọi tiền tố có cùng trạng thái có cùng tập phần hoàn thành.
3. **Cộng `add · cnt`.** Mọi cách hoàn thành từ trạng thái con đều chứa vị trí giữa vừa xét ⇒ mỗi cách nhận thêm `add`. Quy nạp từ cuối: `wav(state) = Σ_y w_suffix(y)` trên mọi hoàn thành `y`.
4. **Số 0 đứng đầu.** Khi chưa `started`, chữ số 0 không được ghi vào `(p2, p1)` ⇒ số ngắn được xử lý như số có ít chữ số hơn (đúng định nghĩa); số 0 không được đếm (`started = False` ở cuối trả `cnt = 0`).
5. `calc(num2) − calc(num1 − 1)` cho tổng trên đoạn.

## 11. Phân tích Time / Space Complexity

Gọi `D = 16` số chữ số:

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O((num2 − num1) · D)` | `O(1)` |
| 2. Digit DP memo | `O(D · 11² · 4 · 10)` | `O(D · 11² · 4)` |
| 3. Digit DP lặp | `O(D · 10² · 10)` | `O(10²)` |
| 4. Đếm đóng góp | `O(D³ · 570)` | `O(1)` |

## 12. Bộ test

In [11]:
FAST = [memo_range, tab_range, lambda a, b: calc_contrib(b) - calc_contrib(a - 1), lambda a, b: Solution().totalWaviness(a, b)]
for a, b, exp in examples:
    assert brute(a, b) == exp and all(f(a, b) == exp for f in FAST)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 tiền tố calc(x) cho mọi x ≤ 3000 và các ca biên
pref = [0]
for y in range(1, 3001): pref.append(pref[-1] + waviness(y))
for x in range(0, 3001):
    assert calc_memo(x) == pref[x] == calc_tab(x), x
for x in [999, 1000, 1001, 9999, 10000]:
    assert calc_contrib(x) == calc_memo(x)
assert Solution().totalWaviness(1, 99) == 0
print("calc(x), x = 0..3000: OK")

calc(x), x = 0..3000: OK


In [13]:
# 12.3 đoạn ngẫu nhiên nhỏ (so vét cạn) và lớn (so giữa các cách nhanh)
rng = random.Random(3753)
for _ in range(300):
    a = rng.randint(1, 200000); b = a + rng.randint(0, 3000)
    assert Solution().totalWaviness(a, b) == brute(a, b)
for _ in range(200):
    a = rng.randint(1, 10**15); b = rng.randint(a, 10**15)
    r = [f(a, b) for f in FAST]
    assert len(set(r)) == 1, (a, b, r)
print("300 đoạn nhỏ + 200 đoạn lớn: OK")

300 đoạn nhỏ + 200 đoạn lớn: OK


In [14]:
# 12.4 thời gian với cận lớn nhất
t0 = time.time(); x = Solution().totalWaviness(1, 10**15); t1 = time.time()
print("tổng trên [1, 10^15] =", x, "| thời gian:", round(t1 - t0, 3), "s")

tổng trên [1, 10^15] = 7360000000000005 | thời gian: 0.005 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi số 0 đứng đầu là chữ số thật

Nếu bỏ `started` và đệm số thành đủ độ dài bằng số 0 ở đầu, `0 5 1` sẽ coi `5` là đỉnh — nhưng `51` chỉ có 2 chữ số.

In [15]:
def wrong_leading_zero(x):
    s = str(x); n = len(s)
    @lru_cache(None)
    def dfs(pos, p2, p1, tight):
        if pos == n: return 1, 0
        up = int(s[pos]) if tight else 9; cnt = wav = 0
        for d in range(up + 1):
            add = 1 if p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)) else 0
            c, w = dfs(pos + 1, p1, d, tight and d == up); cnt += c; wav += w + add * c
        return cnt, wav
    return dfs(0, 10, 10, True)[1]

print("calc(130) sai:", wrong_leading_zero(130), "| đúng:", calc_memo(130))

calc(130) sai: 57 | đúng: 12


### Lỗi 2 — Chỉ trả về tổng (quên nhân với số cách hoàn thành)

In [16]:
def wrong_no_count(x):
    s = str(x); n = len(s)
    @lru_cache(None)
    def dfs(pos, p2, p1, started, tight):
        if pos == n: return 0
        up = int(s[pos]) if tight else 9; wav = 0
        for d in range(up + 1):
            nt = tight and d == up
            if not started:
                wav += dfs(pos + 1, 10, d if d else 10, d > 0, nt)
            else:
                add = 1 if p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)) else 0
                wav += dfs(pos + 1, p1, d, True, nt) + add          # SAI: phải là add * cnt
        return wav
    return dfs(0, 10, 10, False, True)

print("calc(50000) sai:", wrong_no_count(50000), "| đúng:", calc_memo(50000))
assert wrong_no_count(50000) != calc_memo(50000)

calc(50000) sai: 31500 | đúng: 80505


### Lỗi 3 — Dùng `≥`/`≤` (không nghiêm ngặt)

`122` có chữ số giữa `2` **bằng** chữ số sau ⇒ không phải đỉnh.

In [17]:
def wrong_nonstrict(x):
    d = list(map(int, str(x)))
    return sum(1 for i in range(1, len(d) - 1) if (d[i] >= d[i - 1] and d[i] >= d[i + 1]) or (d[i] <= d[i - 1] and d[i] <= d[i + 1]))

print("122 sai:", wrong_nonstrict(122), "| đúng:", waviness(122), "| 111 sai:", wrong_nonstrict(111), "| đúng:", waviness(111))

122 sai: 1 | đúng: 0 | 111 sai: 1 | đúng: 0


### Lỗi 4 — `calc(num1)` thay vì `calc(num1 − 1)`

In [18]:
print("sai:", calc_memo(130) - calc_memo(120), "| đúng:", calc_memo(130) - calc_memo(119))

sai: 2 | đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đoạn số tới `10^15` + tính chất theo **chữ số** ⇒ digit DP với `calc(r) − calc(l − 1)`.
- Hỏi **tổng** một đại lượng (không chỉ đếm) ⇒ DP trả `(cnt, sum)`, chuyển tiếp `sum += add · cnt`.
- Tính chất cục bộ dài `k` chữ số ⇒ lưu `k − 1` chữ số cuối trong trạng thái.
- Luôn xử lý `started` khi tính chất phụ thuộc vị trí/hàng xóm.

## 15. Biến thể của bài

1. **Bản I** (`num2 ≤ 10^5`): vét cạn là đủ.
2. **Chỉ đếm đỉnh** (bỏ đáy): đổi điều kiện `add`.
3. **Đếm số có độ gợn sóng đúng bằng `k`**: thêm chiều `k` vào trạng thái.
4. **Tổng bình phương độ gợn sóng**: DP trả `(cnt, s1, s2)` với `s2 += 2·add·s1 + add²·cnt`.

In [19]:
# Biến thể 3: đếm số trong [1, x] có độ gợn sóng đúng bằng k
def count_exact(x, k):
    s = str(x); n = len(s)
    @lru_cache(None)
    def dfs(pos, p2, p1, started, tight, w):
        if w > k: return 0
        if pos == n: return int(started and w == k)
        up = int(s[pos]) if tight else 9; res = 0
        for d in range(up + 1):
            nt = tight and d == up
            if not started:
                res += dfs(pos + 1, 10, d if d else 10, d > 0, nt, 0)
            else:
                add = 1 if p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)) else 0
                res += dfs(pos + 1, p1, d, True, nt, w + add)
        return res
    return dfs(0, 10, 10, False, True, 0)

for k in range(3):
    assert count_exact(5000, k) == sum(1 for y in range(1, 5001) if waviness(y) == k)
print("Biến thể 3 OK:", [count_exact(5000, k) for k in range(4)])

Biến thể 3 OK: [1205, 2385, 1410, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Từng số | `O(range · D)` | Kiểm chứng, bản I |
| 2. Digit DP memo | `(cnt, wav)` | `O(D · 11² · 40)` | **Lời giải chuẩn** |
| 3. Digit DP lặp | Bảng `(p2, p1)` | `O(D · 1000)` | Không đệ quy |
| 4. Đếm đóng góp | Từng bộ ba cố định | `O(D³ · 570)` | Trực giác, kiểm chứng chéo |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `totalWaviness(100, 110)` = ?

**Bài 2 (tính tay).** Độ gợn sóng của `9090909`.

**Bài 3 (điền TODO).** Viết `total_peaks(num1, num2)` chỉ đếm đỉnh.

**Bài 4 (tự viết lại).** Viết `calc` trả về **tổng bình phương** độ gợn sóng trên `[1, x]`.

**Bài 5 (mini challenge).** Đếm số `y ∈ [1, x]` là "sóng hoàn toàn" (mọi chữ số giữa đều là đỉnh hoặc đáy, và có ít nhất 3 chữ số).

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def total_peaks(num1, num2):
    # TODO: như lời giải, add = 1 chỉ khi p1 > p2 và p1 > d
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_fully_wavy(x):
    # TODO: thêm cờ "còn hợp lệ" vào trạng thái
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Các số `100..110`. (2) Có chữ số giữa nhỏ hơn hẳn hai bên: `101`, `102`, …, `109` (giữa `0` < `1` và < `d`, cần `d > 0`). (3) `101..109` ⇒ 9; `110`: giữa `1` không đỉnh ⇒ tổng `9`.

**Bài 2.** (1) 7 chữ số, 5 vị trí giữa. (2) Luân phiên `0` (đáy) và `9` (đỉnh). (3) `5`.

**Bài 3.** (1) Đổi điều kiện `add`. (2) Giữ phần còn lại. (3) Đối chiếu vét cạn.

**Bài 4.** (1) Trả `(cnt, s1, s2)`. (2) Con có `(c, a, b)`; thêm `add`: `s1 += a + add·c`, `s2 += b + 2·add·a + add²·c`. (3) Kiểm chứng.

**Bài 5.** (1) Cờ `ok`. (2) Ở mỗi bước có đủ 3 chữ số, `ok &= add`. (3) Cuối: `started`, độ dài ≥ 3, `ok`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().totalWaviness(100, 110) == 9 == brute(100, 110)
assert waviness(9090909) == 5
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def total_peaks_sol(num1, num2):
    def calc(x):
        if x <= 0: return 0
        s = str(x); n = len(s)
        @lru_cache(None)
        def dfs(pos, p2, p1, started, tight):
            if pos == n: return (1 if started else 0), 0
            up = int(s[pos]) if tight else 9; cnt = wav = 0
            for d in range(up + 1):
                nt = tight and d == up
                if not started:
                    c, w = dfs(pos + 1, 10, d if d else 10, d > 0, nt); add = 0
                else:
                    add = 1 if p2 != 10 and p1 > p2 and p1 > d else 0
                    c, w = dfs(pos + 1, p1, d, True, nt)
                cnt += c; wav += w + add * c
            return cnt, wav
        return dfs(0, 10, 10, False, True)[1]
    return calc(num2) - calc(num1 - 1)

def calc_sq(x):
    if x <= 0: return 0
    s = str(x); n = len(s)
    @lru_cache(None)
    def dfs(pos, p2, p1, started, tight):
        if pos == n: return (1 if started else 0), 0, 0
        up = int(s[pos]) if tight else 9; C = S1 = S2 = 0
        for d in range(up + 1):
            nt = tight and d == up
            if not started:
                c, a, b = dfs(pos + 1, 10, d if d else 10, d > 0, nt); add = 0
            else:
                add = 1 if p2 != 10 and ((p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)) else 0
                c, a, b = dfs(pos + 1, p1, d, True, nt)
            C += c; S1 += a + add * c; S2 += b + 2 * add * a + add * add * c
        return C, S1, S2
    return dfs(0, 10, 10, False, True)[2]

def peaks(y):
    d = list(map(int, str(y)))
    return sum(1 for i in range(1, len(d) - 1) if d[i] > d[i - 1] and d[i] > d[i + 1])

assert total_peaks_sol(1, 5000) == sum(peaks(y) for y in range(1, 5001))
assert calc_sq(5000) == sum(waviness(y) ** 2 for y in range(1, 5001))
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5
def count_fully_wavy_sol(x):
    s = str(x); n = len(s)
    @lru_cache(None)
    def dfs(pos, p2, p1, length, tight, ok):
        if not ok: return 0
        if pos == n: return int(length >= 3)
        up = int(s[pos]) if tight else 9; res = 0
        for d in range(up + 1):
            nt = tight and d == up
            if length == 0:
                res += dfs(pos + 1, 10, d if d else 10, 1 if d else 0, nt, True)
            else:
                good = True
                if p2 != 10:
                    good = (p1 > p2 and p1 > d) or (p1 < p2 and p1 < d)
                res += dfs(pos + 1, p1, d, min(length + 1, 3), nt, good)
        return res
    return dfs(0, 10, 10, 0, True, True)

def fully_wavy(y):
    s = str(y)
    return len(s) >= 3 and waviness(y) == len(s) - 2

for x in [100, 999, 5000, 12345]:
    assert count_fully_wavy_sol(x) == sum(1 for y in range(1, x + 1) if fully_wavy(y))
print("Bài 5 OK:", count_fully_wavy_sol(10**15))

Bài 5 OK: 1499361366208


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def calc(x):                       # tổng trên [1, x]
    @lru_cache(None)
    def dfs(pos, p2, p1, started, tight):   # trả (cnt, wav)
        ...
        add = p2 != 10 and (p2 < p1 > d or p2 > p1 < d)
        c, w = dfs(pos + 1, p1, d, True, tight and d == up)
        cnt += c; wav += w + add * c
return calc(num2) - calc(num1 - 1)
```

**Checklist:**

- [ ] `calc(num1 − 1)`, không phải `calc(num1)`?
- [ ] Có cờ `started` (số 0 đứng đầu không phải chữ số)?
- [ ] DP trả cả `cnt` và cộng `add · cnt`?
- [ ] So sánh nghiêm ngặt?
- [ ] Đã so với vét cạn trên đoạn nhỏ?